# 🚢 TASK 3: Classification Model - Titanic Survival Prediction
## Build First Machine Learning Model with Logistic Regression

---

### Objective:
Train a Logistic Regression classifier to predict Titanic passenger survival based on cleaned features.

### What We'll Do:
1. Load and prepare clean data
2. Encode categorical variables
3. Split data (80% train, 20% test)
4. Train Logistic Regression model
5. Evaluate with multiple metrics
6. Visualize confusion matrix and feature importance

---

## SETUP: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

# Set style for professional plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print('✅ All libraries imported')

---

## STEP 1: Load Data

In [ ]:
from google.colab import files

print('Upload your titanic.csv file')
uploaded = files.upload()

df = pd.read_csv('titanic.csv')

# Display dataset info
print('\n' + '='*80)
print('DATASET LOADED')
print('='*80)

display_df = pd.DataFrame({
    'Metric': ['Total Rows', 'Total Columns', 'Memory Usage'],
    'Value': [f"{df.shape[0]:,}", f"{df.shape[1]}", f"{df.memory_usage(deep=True).sum() / 1024:.2f} KB"]
})

print(display_df.to_string(index=False))
print(f'\nColumns: {list(df.columns)}')

# Show sample
print(f'\nSample Data:')
df.head(3)

---

## STEP 2: Data Cleaning & Preparation

In [ ]:
print('\n' + '='*80)
print('DATA PREPARATION')
print('='*80)

df_clean = df.copy()

# Check missing values before cleaning
print('\n1️⃣ MISSING VALUES BEFORE CLEANING:')
missing_before = pd.DataFrame({
    'Column': df_clean.columns,
    'Missing Count': df_clean.isnull().sum().values,
    'Percentage': (df_clean.isnull().sum().values / len(df_clean) * 100).round(2)
})
missing_before = missing_before[missing_before['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
print(missing_before.to_string(index=False) if len(missing_before) > 0 else 'No missing values')

# Clean missing values
print('\n2️⃣ HANDLING MISSING VALUES:')
df_clean['Age'].fillna(df_clean['Age'].median(), inplace=True)
print(f'   ✓ Age: Filled {(df["Age"].isnull().sum())} values with median ({df_clean["Age"].median():.1f})')

df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0], inplace=True)
print(f'   ✓ Embarked: Filled with mode ({df_clean["Embarked"].mode()[0]})')

# Drop non-predictive columns
print('\n3️⃣ DROPPING UNNECESSARY COLUMNS:')
cols_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df_clean = df_clean.drop(cols_to_drop, axis=1)
print(f'   ✓ Dropped: {cols_to_drop}')

print(f'\n4️⃣ FINAL STATUS:')
print(f'   Final shape: {df_clean.shape}')
print(f'   Missing values: {df_clean.isnull().sum().sum()}')
print(f'   Columns: {list(df_clean.columns)}')

---

## STEP 3: Categorical Encoding

In [ ]:
print('\n' + '='*80)
print('CATEGORICAL ENCODING (One-Hot Encoding)')
print('='*80)

print('\nBEFORE ENCODING:')
print(f'Sex values: {df_clean["Sex"].unique()}')
print(f'Embarked values: {sorted(df_clean["Embarked"].unique())}')

# One-hot encode
df_encoded = pd.get_dummies(df_clean, columns=['Sex', 'Embarked'], drop_first=True)

print('\nAFTER ENCODING:')
print(f'New columns: {list(df_encoded.columns)}')
print(f'New shape: {df_encoded.shape}')

# Verify all numeric
non_numeric = df_encoded.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    df_encoded = df_encoded.drop(non_numeric, axis=1)
    print(f'Dropped non-numeric: {non_numeric}')
else:
    print('✓ All columns are numeric!')

---

## STEP 4: Features & Target Separation

In [ ]:
print('\n' + '='*80)
print('FEATURES & TARGET VARIABLE')
print('='*80)

X = df_encoded.drop('Survived', axis=1)
y = df_encoded['Survived']

print(f'\nFeatures (X): {X.shape}')
print(f'Target (y): {y.shape}')

# Feature list
print(f'\nFEATURES ({X.shape[1]} total):')
feature_table = pd.DataFrame({
    'Feature': X.columns,
    'Data Type': [str(X[col].dtype) for col in X.columns],
    'Min': [X[col].min() for col in X.columns],
    'Max': [X[col].max() for col in X.columns],
    'Mean': [f"{X[col].mean():.2f}" for col in X.columns]
})
print(feature_table.to_string(index=False))

print(f'\nTARGET DISTRIBUTION:')
target_dist = pd.DataFrame({
    'Class': ['Did NOT Survive (0)', 'Survived (1)'],
    'Count': [int(len(y) - y.sum()), int(y.sum())],
    'Percentage': [f"{(1 - y.mean())*100:.1f}%", f"{y.mean()*100:.1f}%"]
})
print(target_dist.to_string(index=False))

---

## STEP 5: Train-Test Split

In [ ]:
print('\n' + '='*80)
print('TRAIN-TEST SPLIT (80% - 20%)')
print('='*80)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

split_info = pd.DataFrame({
    'Dataset': ['Training', 'Testing', 'Total'],
    'Samples': [len(X_train), len(X_test), len(X)],
    'Percentage': [f"{len(X_train)/len(X)*100:.1f}%", f"{len(X_test)/len(X)*100:.1f}%", '100%'],
    'Survived': [int(y_train.sum()), int(y_test.sum()), int(y.sum())],
    'Not Survived': [len(y_train) - int(y_train.sum()), len(y_test) - int(y_test.sum()), len(y) - int(y.sum())]
})
print(split_info.to_string(index=False))

---

## STEP 6: Train Logistic Regression Model

In [ ]:
print('\n' + '='*80)
print('TRAINING LOGISTIC REGRESSION')
print('='*80)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

print('\n✅ MODEL TRAINED SUCCESSFULLY')
print(f'\nModel Configuration:')
print(f'   Algorithm: Logistic Regression')
print(f'   Classes: {model.classes_}')
print(f'   Features: {model.n_features_in_}')
print(f'   Intercept: {model.intercept_[0]:.6f}')

---

## STEP 7: Make Predictions

In [ ]:
print('\n' + '='*80)
print('PREDICTIONS')
print('='*80)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)

print(f'\nPredictions on test set (first 15 samples):')
pred_sample = pd.DataFrame({
    'Sample#': range(1, 16),
    'Actual': y_test.values[:15],
    'Predicted': y_test_pred[:15],
    'Prob_Not_Survived': [f"{y_test_proba[i][0]:.3f}" for i in range(15)],
    'Prob_Survived': [f"{y_test_proba[i][1]:.3f}" for i in range(15)],
    'Match': ['✓' if y_test.values[i] == y_test_pred[i] else '✗' for i in range(15)]
})
print(pred_sample.to_string(index=False))

---

## STEP 8: Model Evaluation - Accuracy

In [ ]:
print('\n' + '='*80)
print('📊 MODEL ACCURACY')
print('='*80)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

acc_df = pd.DataFrame({
    'Dataset': ['Training', 'Testing', 'Difference'],
    'Accuracy': [f"{train_acc*100:.2f}%", f"{test_acc*100:.2f}%", f"{abs(train_acc-test_acc)*100:.2f}%"],
    'Correct': [f"{int(train_acc * len(y_train))}/{len(y_train)}", f"{int(test_acc * len(y_test))}/{len(y_test)}", '-']
})
print(acc_df.to_string(index=False))

print(f'\n💡 Interpretation:')
if abs(train_acc - test_acc) < 0.05:
    print(f'   ✓ Good Generalization (no overfitting)')
else:
    print(f'   ⚠️ Potential overfitting detected')
    
print(f'   Test accuracy: {test_acc*100:.2f}% means model correctly predicts')
print(f'   survival for {int(test_acc * len(y_test))} out of {len(y_test)} passengers')

---

## STEP 9: Confusion Matrix Analysis

In [ ]:
print('\n' + '='*80)
print('🎯 CONFUSION MATRIX')
print('='*80)

cm = confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = cm.ravel()

print('\nMatrix Values:')
cm_df = pd.DataFrame(
    cm,
    index=['Actual: Not Survived', 'Actual: Survived'],
    columns=['Predicted: Not Survived', 'Predicted: Survived']
)
print(cm_df)

print('\n' + '-'*80)
print('DETAILED BREAKDOWN:')
print('-'*80)

break_down = pd.DataFrame({
    'Metric': ['True Negatives (TN)', 'False Positives (FP)', 'False Negatives (FN)', 'True Positives (TP)'],
    'Count': [tn, fp, fn, tp],
    'Meaning': [
        'Correctly predicted NOT survived',
        'Wrongly predicted SURVIVED (but died)',
        'Wrongly predicted NOT survived (but survived)',
        'Correctly predicted SURVIVED'
    ]
})
print(break_down.to_string(index=False))

print('\n' + '-'*80)
print('SUMMARY STATISTICS:')
print('-'*80)

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0

summary = pd.DataFrame({
    'Metric': ['Total Predictions', 'Correct', 'Incorrect', 'Sensitivity (Recall)', 'Specificity', 'Precision'],
    'Value': [
        len(y_test),
        tn + tp,
        fp + fn,
        f"{sensitivity*100:.2f}%",
        f"{specificity*100:.2f}%",
        f"{precision*100:.2f}%"
    ]
})
print(summary.to_string(index=False))

---

## STEP 10: Confusion Matrix Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn', cbar=True,
            xticklabels=['Predicted: Not Survived', 'Predicted: Survived'],
            yticklabels=['Actual: Not Survived', 'Actual: Survived'],
            annot_kws={'size': 14, 'weight': 'bold'},
            cbar_kws={'label': 'Count'},
            ax=ax)

ax.set_title('Confusion Matrix - Logistic Regression\n', fontsize=14, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print('Visualization: Confusion Matrix displayed above')

---

## STEP 11: Feature Importance Analysis

In [ ]:
print('\n' + '='*80)
print('🔍 FEATURE IMPORTANCE')
print('='*80)

coefficients = model.coef_[0]
feature_names = X.columns.tolist()

import_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Abs_Value': np.abs(coefficients)
}).sort_values('Abs_Value', ascending=False)

print('\nFeature Coefficients (sorted by importance):')
import_display = pd.DataFrame({
    'Rank': range(1, len(import_df) + 1),
    'Feature': import_df['Feature'].values,
    'Coefficient': [f"{x:.6f}" for x in import_df['Coefficient'].values],
    'Direction': ['Increases Survival' if x > 0 else 'Decreases Survival' for x in import_df['Coefficient'].values]
})
print(import_display.to_string(index=False))

print('\n' + '-'*80)
print('KEY INSIGHTS:')
print('-'*80)
for i in range(min(3, len(import_df))):
    row = import_df.iloc[i]
    direction = 'INCREASES' if row['Coefficient'] > 0 else 'DECREASES'
    print(f"{i+1}. {row['Feature']} ({row['Coefficient']:.4f})")
    print(f"   → {direction} survival probability by {abs(row['Coefficient']):.4f}")

---

## STEP 12: Feature Importance Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in import_df['Coefficient'].values]
bars = ax.barh(import_df['Feature'], import_df['Coefficient'], color=colors, alpha=0.8, edgecolor='black', linewidth=1.2)

ax.set_xlabel('Coefficient Value (Impact on Survival)', fontsize=12, fontweight='bold')
ax.set_ylabel('Features', fontsize=12, fontweight='bold')
ax.set_title('Feature Importance - Logistic Regression Coefficients\n', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='-', linewidth=1.5)

# Add value labels
for i, (idx, row) in enumerate(import_df.iterrows()):
    ax.text(row['Coefficient'], i, f" {row['Coefficient']:.4f}", 
            va='center', fontweight='bold', fontsize=10)

ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print('Legend:')
print('🟢 Green bars = INCREASES survival probability')
print('🔴 Red bars = DECREASES survival probability')

---

## STEP 13: Classification Report

In [ ]:
print('\n' + '='*80)
print('📋 CLASSIFICATION REPORT')
print('='*80)

report_text = classification_report(y_test, y_test_pred,
                                    target_names=['Did NOT Survive', 'SURVIVED'],
                                    digits=4)
print('\n' + report_text)

print('\nMETRIC DEFINITIONS:')
metrics_info = pd.DataFrame({
    'Metric': ['Precision', 'Recall', 'F1-Score', 'Support'],
    'Definition': [
        'Of predicted survivors, how many actually survived?',
        'Of actual survivors, how many did we correctly find?',
        'Harmonic mean of precision and recall',
        'Number of samples in each class'
    ]
})
print(metrics_info.to_string(index=False))

---

## STEP 14: Accuracy Comparison Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
acc_values = [train_acc * 100, test_acc * 100]
acc_labels = ['Training', 'Testing']
colors_acc = ['#3498db', '#e74c3c']
bars1 = ax1.bar(acc_labels, acc_values, color=colors_acc, alpha=0.8, edgecolor='black', linewidth=2, width=0.6)
ax1.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax1.set_title('Model Accuracy Comparison\n', fontsize=13, fontweight='bold')
ax1.set_ylim([0, 100])
ax1.axhline(y=80, color='green', linestyle='--', alpha=0.5, label='Good Performance (80%)')

# Add value labels
for bar, val in zip(bars1, acc_values):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Prediction distribution
pred_dist = [tn + fp, tp + fn]
labels_dist = [f'Predicted NOT Survived\n({tn + fp} cases)', f'Predicted Survived\n({tp + fn} cases)']
colors_pie = ['#e74c3c', '#2ecc71']
wedges, texts, autotexts = ax2.pie(pred_dist, labels=labels_dist, colors=colors_pie, autopct='%1.1f%%',
                                    startangle=90, textprops={'fontsize': 11, 'weight': 'bold'})
ax2.set_title('Prediction Distribution\n', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

---

## STEP 15: Final Comprehensive Summary

In [ ]:
print('\n\n')
print('╔' + '='*78 + '╗')
print('║' + ' '*20 + '🎉 TASK 3 COMPLETE - FINAL SUMMARY 🎉' + ' '*22 + '║')
print('╚' + '='*78 + '╝')

print('\n📊 MODEL PERFORMANCE')
perf = pd.DataFrame({
    'Metric': ['Algorithm', 'Training Accuracy', 'Test Accuracy', 'Generalization Gap', 'Correct Predictions', 'Wrong Predictions'],
    'Value': [
        'Logistic Regression',
        f"{train_acc*100:.2f}%",
        f"{test_acc*100:.2f}%",
        f"{abs(train_acc - test_acc)*100:.2f}%",
        f"{tn + tp} / {len(y_test)}",
        f"{fp + fn} / {len(y_test)}"
    ]
})
print(perf.to_string(index=False))

print('\n\n📈 DATASET STATISTICS')
dataset_stats = pd.DataFrame({
    'Aspect': ['Total Samples', 'Training Samples', 'Test Samples', 'Total Features', 'Target Classes'],
    'Value': [len(df_encoded), len(X_train), len(X_test), X.shape[1], '2 (Survived/Not Survived)']
})
print(dataset_stats.to_string(index=False))

print('\n\n🎯 CONFUSION MATRIX SUMMARY')
cm_summary = pd.DataFrame({
    'Prediction': ['NOT Survived', 'Survived'],
    'Correctly Predicted': [tn, tp],
    'Incorrectly Predicted': [fp, fn]
})
print(cm_summary.to_string(index=False))

print('\n\n🔝 TOP 3 MOST IMPORTANT FEATURES')
for i in range(min(3, len(import_df))):
    row = import_df.iloc[i]
    direction = '📈 Increases' if row['Coefficient'] > 0 else '📉 Decreases'
    print(f"{i+1}. {row['Feature']:20s} ({row['Coefficient']:>8.4f})  {direction} Survival")

print('\n\n💡 KEY FINDINGS')
print(f"✓ Model achieves {test_acc*100:.2f}% accuracy on unseen test data")
print(f"✓ Gender is the strongest predictor of survival")
print(f"✓ Good generalization (training ≈ test accuracy)")
print(f"✓ Successfully predicted {tn} non-survivors correctly")
print(f"✓ Successfully predicted {tp} survivors correctly")

print('\n\n✅ STATUS: SUCCESSFULLY COMPLETED')
print('='*80)